# 00 · Pipeline and inventory

**Purpose.** Overview of the whole chain and a check that every file it needs is there, readable and produced with the same settings.

**Inputs.** nexus files in `$NEXUS_DIR`, the 2025 IC outputs in `$OLD_IC_DIR` and the new outputs in `$ICA_OUT`.

**Outputs.** None (tables printed here).

**What is checked.** file readability (first and last row of every table), and the detsim settings stored inside each output: detector, S1 and S2 tables, `el_gain`, lifetime, and the nexus file it came from.

**Next step.** `01_nexus_inputs`.

## The pipeline

| Step | Program | Input | Output | Notebook |
|---|---|---|---|---|
| 0 | nexus (Geant4), EL off | macro | `<sample>_<i>.next.h5`: true energy deposits | `01_nexus_inputs` |
| – | light tables | nexus ie runs | S2 table per sensor (fibers: `20250408_s2_table.h5`; PMTs: IC `NEXT100_S2_LT.h5`) | `02_light_tables` |
| 1 | IC **detsim** | nexus file + tables | `.detsim.h5`: raw waveforms (`/fibrd` or `/pmtrd`, `/sipmrd`) | `03_detsim` |
| 2 | IC **hypathia** | detsim | `.hypathia.h5`: PMaps (S1, S2 peaks) | `04_hypathia` |
| 3 | IC **dorothea** | hypathia | `.dorothea.h5`: kDST, one row per S1-S2 pair (Kr) | `05_dorothea_kr` |
| 4 | Q_bb energy | hypathia (0νββ) | energy per event | `06_qbb` |

Run a step with `./submit.sh <city> <fibers|pmts> <sample>` (dry run) and add `--submit` to send it.
New outputs go to `$ICA_OUT/<sample>/<fibers|pmts>/<city>/`. The 2025 outputs are never overwritten.

In [ ]:
# Setup: run `source env.sh` before starting Jupyter, or pick the IC-3.8-2024-06-08 kernel.
import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks, fits
print("ICDIR =", paths.ICDIR or "NOT SET (source env.sh)")

## Samples (nexus inputs)

In [ ]:
samples = {"20250624_Next100_1Kev_ELoff_Kr": "Kr, matched fibers/PMTs",
           "20250626_Next100_50Kev_ELoff_Kr": "Kr, 2025 fibers result",
           "20250205_Next100_bb_300ev_ELoff": "0nubb"}
rows = []
for s, what in samples.items():
    fs = paths.nexus_files(s)
    ok = [checks.is_readable(f)[0] for f in fs]
    cfg = checks.nexus_config(fs[0]) if fs else {}
    rows.append(dict(sample=s, what=what, files=len(fs), readable=sum(ok),
                     unreadable=[paths.file_index(f) for f, o in zip(fs, ok) if not o], **cfg))
nexus_inv = pd.DataFrame(rows); nexus_inv

## 2025 IC outputs: readable, and produced with the same settings?

In [ ]:
rows = []
for s in ["20250624_Next100_1Kev_ELoff_Kr", "20250626_Next100_50Kev_ELoff_Kr"]:
    for det in ["fibers", "pmts"]:
        for city in ["detsim", "hypathia", "dorothea"]:
            fs = paths.old_files(s, det, city)
            if not fs: continue
            t = checks.summary_ic(fs)
            good = t[t.readable]
            rows.append(dict(sample=s, det=det, city=city, files=len(t), readable=int(t.readable.sum()),
                             unreadable=[f for f in t.file[~t.readable]],
                             detector_db=sorted(good.detector_db.dropna().unique()) if "detector_db" in good else [],
                             s2_table=sorted(good.s2_table.dropna().unique()) if "s2_table" in good else [],
                             el_gain=sorted(good.el_gain.dropna().unique()) if "el_gain" in good else []))
old_inv = pd.DataFrame(rows); old_inv

In [ ]:
# Checks: one detector, one S2 table and one el_gain per (sample, detector)
for _, r in old_inv.iterrows():
    same = len(r.detector_db) <= 1 and len(r.el_gain) <= 1 and len({os.path.basename(t) for t in r.s2_table}) <= 1
    print(f"{'OK  ' if same else 'DIFF'} {r['sample']:34s} {r.det:6s} {r.city:8s} "
          f"readable {r.readable}/{r.files}  tables {sorted({os.path.basename(t) for t in r.s2_table})}  el_gain {r.el_gain}")

## New outputs of this repository

In [ ]:
for s in samples:
    for det in ["fibers", "pmts"]:
        for city in ["detsim", "hypathia", "dorothea"]:
            fs = paths.new_files(s, det, city)
            if fs: print(f"{s:34s} {det:6s} {city:8s} {len(fs)} files")
print("(nothing listed = nothing run yet)")

## Notes
- Unreadable 2025 files are listed above (cluster storage incident, `~/corrupted_files.txt`, Dec 2025). They can be regenerated from the previous step with `submit.sh`.
- `el_gain = 365` is the IC default; nexus at 10 bar and 13 kV/cm gives 1050 ph/e. The fiber/PMT ratio does not depend on it; absolute pes do.